# Coastal Flood Step 17: Mangrove Priority Ranking (Protection-Shed Outputs)

This notebook prioritizes mangrove patches for avoided-damage benefits using the **minimum + maximum scenario protection-shed attribution outputs**.

Outputs:
- Full ranked table
- No-regret shortlist
- Tiered priority map


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import substring
from shapely.geometry import MultiLineString

pd.set_option('display.max_columns', 200)


In [ ]:
# -----------------------------
# Parameters
# -----------------------------
TOP_N = 25
TOP_QUARTILE_SHARE = 0.25
TIER1_SHARE = 0.20
TIER2_SHARE = 0.30

# Priority score weights
W_TOTAL = 0.55
W_EFF = 0.30
W_ROBUST = 0.15

# Protection-shed geometry params (kept aligned with notebook 16)
COAST_SEGMENT_LENGTH_M = 2000.0
MANGROVE_CONNECT_BUFFER_M = 500.0

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')

min_results_dir = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario'
max_results_dir = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario'

min_dir = min_results_dir / 'damage_estimates/mangrove_attribution_protection_shed'
max_dir = max_results_dir / 'damage_estimates/mangrove_attribution_protection_shed'

min_damage_dir = min_results_dir / 'damage_estimates'
max_damage_dir = max_results_dir / 'damage_estimates'

min_gpkg = min_dir / 'mangrove_attribution_total_protection_shed_minimum.gpkg'
max_csv = max_dir / 'mangrove_attribution_total_protection_shed_maximum.csv'

boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

out_dir = base_path / 'dphil_paper_3/results/mangrove_priority_protection_shed'
out_dir.mkdir(parents=True, exist_ok=True)

print('Input minimum:', min_gpkg)
print('Input maximum:', max_csv)
print('Output folder:', out_dir)


In [ ]:
# Load attribution outputs + compute increased-damage flags by mangrove patch
if not min_gpkg.exists():
    raise FileNotFoundError(f'Missing file: {min_gpkg}')
if not max_csv.exists():
    raise FileNotFoundError(f'Missing file: {max_csv}')
if not boundary_path.exists():
    raise FileNotFoundError(f'Missing file: {boundary_path}')

min_gdf = gpd.read_file(min_gpkg)
max_df = pd.read_csv(max_csv)

if str(min_gdf.crs).upper() != 'EPSG:3448':
    min_gdf = min_gdf.to_crs('EPSG:3448')

max_df = max_df.rename(columns={'Total_Avoided_EAD_USD_attributed': 'avoided_usd_max'})

priority = min_gdf.rename(columns={'Total_Avoided_EAD_USD_attributed': 'avoided_usd_min'}).merge(
    max_df[['Mangrove_ID', 'avoided_usd_max']],
    on='Mangrove_ID',
    how='left'
)

priority['avoided_usd_min'] = priority['avoided_usd_min'].fillna(0.0)
priority['avoided_usd_max'] = priority['avoided_usd_max'].fillna(0.0)

if 'capacity_weight_raw' not in priority.columns:
    area = priority.geometry.area
    perim = priority.geometry.length.replace(0, np.nan)
    eff_width = (2.0 * area / perim).fillna(0.0)
    priority['capacity_weight_raw'] = area * eff_width.clip(lower=1.0)

print('Patches loaded:', len(priority))

boundary = gpd.read_file(boundary_path)
if str(boundary.crs).upper() != 'EPSG:3448':
    boundary = boundary.to_crs('EPSG:3448')

def split_line_into_segments(line, segment_length):
    if line.is_empty or line.length == 0:
        return []
    n = max(1, int(np.ceil(line.length / segment_length)))
    distances = np.linspace(0, line.length, n + 1)
    out = []
    for i in range(n):
        seg = substring(line, float(distances[i]), float(distances[i + 1]))
        if not seg.is_empty and seg.length > 0:
            out.append(seg)
    return out

def infer_id_column(cols):
    preferred = ['edge_id', 'node_id', 'id', 'osm_id']
    for c in preferred:
        if c in cols:
            return c
    fallback = [c for c in cols if c.endswith('_id')]
    if fallback:
        return fallback[0]
    raise KeyError(f'Could not infer ID column from columns: {list(cols)}')

def build_segment_patch_map(mangroves_gdf, boundary_gdf):
    coast_geom = boundary_gdf.geometry.iloc[0].boundary
    line_geoms = list(coast_geom.geoms) if isinstance(coast_geom, MultiLineString) else [coast_geom]

    segments = []
    for ln in line_geoms:
        segments.extend(split_line_into_segments(ln, COAST_SEGMENT_LENGTH_M))

    coast_segments = gpd.GeoDataFrame(
        {'segment_id': np.arange(1, len(segments) + 1)},
        geometry=segments,
        crs='EPSG:3448'
    )

    mangrove_buffers = mangroves_gdf[['Mangrove_ID', 'capacity_weight_raw', 'geometry']].copy()
    mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(MANGROVE_CONNECT_BUFFER_M)

    seg_to_patch = gpd.sjoin(
        coast_segments[['segment_id', 'geometry']],
        mangrove_buffers[['Mangrove_ID', 'capacity_weight_raw', 'geometry']],
        how='left',
        predicate='intersects'
    )[['segment_id', 'Mangrove_ID', 'capacity_weight_raw']]

    covered_seg_ids = set(seg_to_patch.loc[seg_to_patch['Mangrove_ID'].notna(), 'segment_id'].astype(int).unique())
    missing_seg_ids = set(coast_segments['segment_id']) - covered_seg_ids

    if missing_seg_ids:
        missing_seg = coast_segments[coast_segments['segment_id'].isin(sorted(missing_seg_ids))].copy()
        nearest_fill = gpd.sjoin_nearest(
            missing_seg[['segment_id', 'geometry']],
            mangroves_gdf[['Mangrove_ID', 'capacity_weight_raw', 'geometry']],
            how='left',
            distance_col='dist_to_mangrove_m'
        )[['segment_id', 'Mangrove_ID', 'capacity_weight_raw']]
        seg_to_patch = pd.concat([seg_to_patch, nearest_fill], ignore_index=True)

    seg_to_patch = seg_to_patch.dropna(subset=['segment_id', 'Mangrove_ID']).copy()
    seg_to_patch['segment_id'] = seg_to_patch['segment_id'].astype(int)
    seg_to_patch['Mangrove_ID'] = seg_to_patch['Mangrove_ID'].astype(int)

    seg_to_patch['weight_raw'] = seg_to_patch['capacity_weight_raw'].fillna(0).clip(lower=1.0)
    seg_to_patch['weight_sum'] = seg_to_patch.groupby('segment_id')['weight_raw'].transform('sum')
    seg_to_patch['segment_patch_weight'] = seg_to_patch['weight_raw'] / seg_to_patch['weight_sum']

    return coast_segments, seg_to_patch[['segment_id', 'Mangrove_ID', 'segment_patch_weight']]

def attribute_increased_damage(damage_dir, coast_segments, seg_to_patch, scenario_label):
    asset_ead_csv = damage_dir / 'coastal_ead_asset_level_usd.csv'
    if not asset_ead_csv.exists():
        print(f'[{scenario_label}] Missing file: {asset_ead_csv}')
        return pd.DataFrame(columns=['Mangrove_ID', f'increase_usd_{scenario_label}'])

    asset_ead = pd.read_csv(asset_ead_csv)
    negative = asset_ead.loc[asset_ead['Avoided_EAD_USD'] < 0].copy()

    if negative.empty:
        print(f'[{scenario_label}] No negative avoided values found.')
        return pd.DataFrame(columns=['Mangrove_ID', f'increase_usd_{scenario_label}'])

    negative['Asset_ID'] = negative['Asset_ID'].astype(str)

    parts = []
    missing_geom_files = []

    for (asset_name, layer_name), sub in negative.groupby(['Asset', 'Layer'], dropna=False):
        gpkg_path = damage_dir / f'{asset_name}_{layer_name}_asset_damages_groupedby.gpkg'
        if not gpkg_path.exists():
            missing_geom_files.append(gpkg_path.name)
            continue

        gdf = gpd.read_file(gpkg_path)
        id_col = infer_id_column(gdf.columns)

        geom_df = gdf[[id_col, 'geometry']].copy()
        geom_df['Asset_ID'] = geom_df[id_col].astype(str)
        geom_df = geom_df.drop(columns=[id_col])
        geom_df = geom_df.dropna(subset=['geometry'])

        geom_df['geometry'] = geom_df.geometry.representative_point()
        geom_df = geom_df.drop_duplicates(subset=['Asset_ID'])

        merged = sub.merge(geom_df[['Asset_ID', 'geometry']], on='Asset_ID', how='left')
        merged = merged.dropna(subset=['geometry']).copy()
        if merged.empty:
            continue

        part = gpd.GeoDataFrame(merged, geometry='geometry', crs=gdf.crs)
        if str(part.crs).upper() != 'EPSG:3448':
            part = part.to_crs('EPSG:3448')

        parts.append(part)

    if not parts:
        print(f'[{scenario_label}] No negative rows with geometry were found.')
        return pd.DataFrame(columns=['Mangrove_ID', f'increase_usd_{scenario_label}'])

    asset_points = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry='geometry', crs='EPSG:3448')

    asset_with_segment = gpd.sjoin_nearest(
        asset_points,
        coast_segments[['segment_id', 'geometry']],
        how='left',
        distance_col='dist_to_segment_m'
    )

    asset_with_segment = asset_with_segment.dropna(subset=['segment_id']).copy()
    asset_with_segment['segment_id'] = asset_with_segment['segment_id'].astype(int)
    asset_with_segment = asset_with_segment.drop(columns=['index_right'])

    asset_patch = asset_with_segment.merge(
        seg_to_patch[['segment_id', 'Mangrove_ID', 'segment_patch_weight']],
        on='segment_id',
        how='left'
    )

    asset_patch = asset_patch.dropna(subset=['Mangrove_ID', 'segment_patch_weight']).copy()
    asset_patch['Mangrove_ID'] = asset_patch['Mangrove_ID'].astype(int)

    asset_patch['Avoided_EAD_USD_attributed'] = asset_patch['Avoided_EAD_USD'] * asset_patch['segment_patch_weight']
    asset_patch[f'increase_usd_{scenario_label}'] = (-asset_patch['Avoided_EAD_USD_attributed']).clip(lower=0.0)

    increased = (
        asset_patch.groupby('Mangrove_ID', as_index=False)[f'increase_usd_{scenario_label}']
        .sum()
    )

    print(
        f'[{scenario_label}] negative rows: {len(negative):,} | '
        f'rows with geometry: {len(asset_points):,} | '
        f'patches with increase: {(increased[f"increase_usd_{scenario_label}"] > 0).sum():,} | '
        f'total increased USD: {increased[f"increase_usd_{scenario_label}"].sum():,.2f}'
    )
    if missing_geom_files:
        print(f'[{scenario_label}] Missing geometry files skipped: {sorted(set(missing_geom_files))}')

    return increased

coast_segments, seg_to_patch = build_segment_patch_map(
    priority[['Mangrove_ID', 'capacity_weight_raw', 'geometry']].copy(),
    boundary
)

inc_min = attribute_increased_damage(min_damage_dir, coast_segments, seg_to_patch, 'min')
inc_max = attribute_increased_damage(max_damage_dir, coast_segments, seg_to_patch, 'max')

priority = priority.merge(inc_min, on='Mangrove_ID', how='left')
priority = priority.merge(inc_max, on='Mangrove_ID', how='left')

priority['increase_usd_min'] = priority['increase_usd_min'].fillna(0.0)
priority['increase_usd_max'] = priority['increase_usd_max'].fillna(0.0)

priority['has_increase_min'] = priority['increase_usd_min'] > 0
priority['has_increase_max'] = priority['increase_usd_max'] > 0
priority['has_increase_any'] = priority['has_increase_min'] | priority['has_increase_max']


In [ ]:
# Metrics
priority['area_m2'] = priority.geometry.area
priority['area_ha'] = priority['area_m2'] / 10000.0

priority['avoided_usd_mean'] = (priority['avoided_usd_min'] + priority['avoided_usd_max']) / 2.0
priority['avoided_usd_range'] = priority['avoided_usd_max'] - priority['avoided_usd_min']

priority['usd_per_ha_min'] = np.where(priority['area_ha'] > 0, priority['avoided_usd_min'] / priority['area_ha'], np.nan)
priority['usd_per_ha_max'] = np.where(priority['area_ha'] > 0, priority['avoided_usd_max'] / priority['area_ha'], np.nan)
priority['usd_per_ha_mean'] = np.where(priority['area_ha'] > 0, priority['avoided_usd_mean'] / priority['area_ha'], np.nan)

# Ranks (1 = best)
priority['rank_min'] = priority['avoided_usd_min'].rank(method='dense', ascending=False)
priority['rank_max'] = priority['avoided_usd_max'].rank(method='dense', ascending=False)
priority['rank_mean'] = priority['avoided_usd_mean'].rank(method='dense', ascending=False)
priority['rank_eff'] = priority['usd_per_ha_mean'].rank(method='dense', ascending=False)
priority['rank_volatility'] = (priority['rank_min'] - priority['rank_max']).abs()

n = len(priority)
if n <= 1:
    priority['robustness_norm'] = 1.0
else:
    priority['robustness_norm'] = 1.0 - (priority['rank_volatility'] / (n - 1))

# Normalize drivers for priority score

def minmax(s):
    mn, mx = float(s.min()), float(s.max())
    if np.isclose(mx, mn):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

priority['total_norm'] = minmax(priority['avoided_usd_mean'])
priority['eff_norm'] = minmax(priority['usd_per_ha_mean'].fillna(0.0))

priority['priority_score'] = (
    W_TOTAL * priority['total_norm'] +
    W_EFF * priority['eff_norm'] +
    W_ROBUST * priority['robustness_norm']
)

priority['priority_rank'] = priority['priority_score'].rank(method='dense', ascending=False)


In [ ]:
# No-regret and tier flags
n = len(priority)
q_cut = max(1, int(np.ceil(TOP_QUARTILE_SHARE * n)))

def tier_from_rank(r):
    t1 = max(1, int(np.ceil(TIER1_SHARE * n)))
    t2 = max(t1 + 1, int(np.ceil((TIER1_SHARE + TIER2_SHARE) * n)))
    if r <= t1:
        return 'Tier 1 (Highest)'
    elif r <= t2:
        return 'Tier 2 (Medium)'
    return 'Tier 3 (Lower)'

priority['no_regret'] = (
    (priority['rank_min'] <= q_cut) &
    (priority['rank_max'] <= q_cut) &
    (priority['avoided_usd_min'] > 0) &
    (priority['avoided_usd_max'] > 0)
)

priority['priority_tier'] = priority['priority_rank'].apply(tier_from_rank)

priority = priority.sort_values(['priority_rank', 'rank_mean', 'Mangrove_ID']).reset_index(drop=True)

cols_show = [
    'Mangrove_ID', 'priority_rank', 'priority_tier', 'no_regret',
    'avoided_usd_min', 'avoided_usd_max', 'avoided_usd_mean', 'avoided_usd_range',
    'increase_usd_min', 'increase_usd_max', 'has_increase_any',
    'usd_per_ha_mean', 'rank_min', 'rank_max', 'rank_mean', 'rank_volatility', 'priority_score'
]

display(priority[cols_show].head(TOP_N))
print('Patches with increased damage in min or max:', int(priority['has_increase_any'].sum()))


In [ ]:
# Save outputs
full_csv = out_dir / 'mangrove_priority_full_table_protection_shed.csv'
short_csv = out_dir / f'mangrove_priority_top_{TOP_N}_protection_shed.csv'
noregret_csv = out_dir / 'mangrove_priority_no_regret_protection_shed.csv'
increase_csv = out_dir / 'mangrove_priority_increased_damage_flags_protection_shed.csv'
map_gpkg = out_dir / 'mangrove_priority_map_protection_shed.gpkg'

priority.to_csv(full_csv, index=False)
priority.head(TOP_N).to_csv(short_csv, index=False)
priority.loc[priority['no_regret']].to_csv(noregret_csv, index=False)
priority[[
    'Mangrove_ID', 'increase_usd_min', 'increase_usd_max',
    'has_increase_min', 'has_increase_max', 'has_increase_any'
]].to_csv(increase_csv, index=False)
priority.to_file(map_gpkg, driver='GPKG')

print('Saved:')
print(' -', full_csv)
print(' -', short_csv)
print(' -', noregret_csv)
print(' -', increase_csv)
print(' -', map_gpkg)


In [ ]:
# Summary stats
summary = {
    'total_patches': len(priority),
    'total_avoided_usd_min': float(priority['avoided_usd_min'].sum()),
    'total_avoided_usd_max': float(priority['avoided_usd_max'].sum()),
    'total_avoided_usd_mean': float(priority['avoided_usd_mean'].sum()),
    'no_regret_count': int(priority['no_regret'].sum()),
    'tier1_count': int((priority['priority_tier'] == 'Tier 1 (Highest)').sum()),
    'tier2_count': int((priority['priority_tier'] == 'Tier 2 (Medium)').sum()),
    'tier3_count': int((priority['priority_tier'] == 'Tier 3 (Lower)').sum()),
}

pd.Series(summary)


In [ ]:
# Plot: top-N bar chart
plot_df = priority.head(TOP_N).copy()
plot_df = plot_df.sort_values('priority_rank', ascending=False)

fig, ax = plt.subplots(figsize=(9, max(6, TOP_N * 0.22)))
ax.barh(plot_df['Mangrove_ID'].astype(str), plot_df['avoided_usd_mean'], color='#2f4f4f')
ax.set_title(f'Top {TOP_N} Mangrove Patches by Mean Avoided EAD (USD)')
ax.set_xlabel('Mean avoided EAD (USD)')
ax.set_ylabel('Mangrove ID')
plt.tight_layout()

bar_png = out_dir / f'mangrove_priority_top_{TOP_N}_bar.png'
fig.savefig(bar_png, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', bar_png)


In [ ]:
# Plot: tiered maps (minimum and maximum increased damage shown separately)
from matplotlib.patches import Patch

boundary = gpd.read_file(boundary_path)
if str(boundary.crs).upper() != 'EPSG:3448':
    boundary = boundary.to_crs('EPSG:3448')

tier_colors = {
    'Tier 1 (Highest)': '#2E7D32',
    'Tier 2 (Medium)': '#F9A825',
    'Tier 3 (Lower)': '#B0BEC5'
}

def plot_tier_map_with_increase(increase_col, increase_label, out_name, title):
    fig, ax = plt.subplots(figsize=(8, 10))
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.8)

    for tier, color in tier_colors.items():
        subset = priority.loc[priority['priority_tier'] == tier]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1)

    increased = priority.loc[priority[increase_col]]
    if len(increased) > 0:
        increased.plot(
            ax=ax,
            facecolor='none',
            edgecolor='#C62828',
            linewidth=0.9,
            hatch='////'
        )

    legend_handles = [
        Patch(facecolor=tier_colors[t], edgecolor='black', label=t)
        for t in ['Tier 1 (Highest)', 'Tier 2 (Medium)', 'Tier 3 (Lower)']
    ]
    legend_handles.append(
        Patch(facecolor='white', edgecolor='#C62828', hatch='////', label=increase_label)
    )

    ax.set_title(title)
    ax.set_axis_off()
    ax.legend(handles=legend_handles, loc='lower left', frameon=True, title='Map Layers')
    plt.tight_layout()

    out_png = out_dir / out_name
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', out_png)

plot_tier_map_with_increase(
    increase_col='has_increase_min',
    increase_label='Increased damage (minimum scenario)',
    out_name='mangrove_priority_tier_map_protection_shed_minimum.png',
    title='Mangrove Priority Tiers + Increased-Damage Hotspots (Minimum)'
)

plot_tier_map_with_increase(
    increase_col='has_increase_max',
    increase_label='Increased damage (maximum scenario)',
    out_name='mangrove_priority_tier_map_protection_shed_maximum.png',
    title='Mangrove Priority Tiers + Increased-Damage Hotspots (Maximum)'
)


In [ ]:
# Plot: net effect maps (net positive vs net negative), minimum and maximum separately
from matplotlib.patches import Patch

boundary = gpd.read_file(boundary_path)
if str(boundary.crs).upper() != 'EPSG:3448':
    boundary = boundary.to_crs('EPSG:3448')

priority['net_usd_min'] = priority['avoided_usd_min'] - priority['increase_usd_min']
priority['net_usd_max'] = priority['avoided_usd_max'] - priority['increase_usd_max']

def classify_net(x):
    if x > 0:
        return 'Net positive'
    if x < 0:
        return 'Net negative'
    return 'Neutral (0)'

priority['net_class_min'] = priority['net_usd_min'].apply(classify_net)
priority['net_class_max'] = priority['net_usd_max'].apply(classify_net)

net_colors = {
    'Net positive': '#2E7D32',
    'Net negative': '#C62828',
    'Neutral (0)': '#CFD8DC'
}

def plot_net_map(class_col, out_name, title):
    fig, ax = plt.subplots(figsize=(8, 10))
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.8)

    for cls in ['Neutral (0)', 'Net positive', 'Net negative']:
        subset = priority.loc[priority[class_col] == cls]
        if len(subset) > 0:
            subset.plot(ax=ax, color=net_colors[cls], edgecolor='black', linewidth=0.1)

    handles = [
        Patch(facecolor=net_colors['Net positive'], edgecolor='black', label='Net positive (avoided > increased)'),
        Patch(facecolor=net_colors['Net negative'], edgecolor='black', label='Net negative (increased > avoided)'),
        Patch(facecolor=net_colors['Neutral (0)'], edgecolor='black', label='Neutral (equal/zero)')
    ]

    ax.set_title(title)
    ax.set_axis_off()
    ax.legend(handles=handles, loc='lower left', frameon=True, title='Net Effect')
    plt.tight_layout()

    out_png = out_dir / out_name
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', out_png)

plot_net_map(
    class_col='net_class_min',
    out_name='mangrove_priority_net_effect_map_minimum.png',
    title='Mangrove Patch Net Flood-Risk Effect (Minimum)'
)

plot_net_map(
    class_col='net_class_max',
    out_name='mangrove_priority_net_effect_map_maximum.png',
    title='Mangrove Patch Net Flood-Risk Effect (Maximum)'
)

print('Minimum map counts:', priority['net_class_min'].value_counts().to_dict())
print('Maximum map counts:', priority['net_class_max'].value_counts().to_dict())
